# Universal Precision Runtime (UPR) — Notebook 01
## BitPlane Model Conversion & Level 1 Weight Reconstruction Verification

---

### Objective
1. Download original FP16 baseline model (`Qwen/Qwen3.5-0.8B`).
2. Convert every parameter tensor into packed bit-plane files (`plane15.bin` through `plane0.bin`).
3. Reconstruct full 16-bit FP16 weights from bit-planes.
4. Validate Stage 1 Success Criterion: **100% exact bitwise tensor equality (`torch.equal == True`)** across all parameters.

### Step 1: Environment Setup & Automatic Package Deployment
Mount Google Drive (for persisting checkpoints), set up Hugging Face authentication, and initialize `upr` package imports.

In [1]:
import os
import sys

# Set Hugging Face Token
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

# Mount Google Drive if in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
    DRIVE_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
except ImportError:
    print('Running in local environment.')

WORK_DIR = os.getcwd()
print(f"Active Working Directory: {WORK_DIR}")
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Automatic Bootstrap: Write UPR package files to disk
os.makedirs("upr", exist_ok=True)

with open("upr/__init__.py", "w", encoding="utf-8") as f:
    f.write('''from .bit_ops import (
    float16_to_uint16_numpy,
    uint16_to_float16_torch,
    extract_bit_plane_np,
    pack_bit_plane,
    unpack_bit_plane,
    reconstruct_tensor,
)
from .converter import convert_to_bitplanes
from .loader import BitPlaneModel
from .metrics import compute_weight_metrics

__version__ = "0.1.0"
__all__ = [
    "float16_to_uint16_numpy",
    "uint16_to_float16_torch",
    "extract_bit_plane_np",
    "pack_bit_plane",
    "unpack_bit_plane",
    "reconstruct_tensor",
    "convert_to_bitplanes",
    "BitPlaneModel",
    "compute_weight_metrics",
]
''')

with open("upr/bit_ops.py", "w", encoding="utf-8") as f:
    f.write('''import torch
import numpy as np
from typing import Tuple, Dict, Optional, Union

def float16_to_uint16_numpy(tensor: torch.Tensor) -> np.ndarray:
    np_f16 = tensor.detach().cpu().to(torch.float16).numpy()
    return np_f16.view(np.uint16)

def uint16_to_float16_torch(np_uint16: np.ndarray, device: Union[str, torch.device] = 'cpu') -> torch.Tensor:
    np_f16 = np_uint16.view(np.float16)
    return torch.from_numpy(np_f16).to(device)

def extract_bit_plane_np(uint16_arr: np.ndarray, bit_index: int) -> np.ndarray:
    assert 0 <= bit_index <= 15, f"bit_index must be between 0 and 15, got {bit_index}"
    return ((uint16_arr >> bit_index) & 1).astype(np.uint8)

def pack_bit_plane(bit_arr: np.ndarray) -> bytes:
    flat = bit_arr.ravel()
    packed = np.packbits(flat, bitorder='big')
    return packed.tobytes()

def unpack_bit_plane(packed_bytes: bytes, num_elements: int, shape: Optional[Tuple[int, ...]] = None) -> np.ndarray:
    packed_np = np.frombuffer(packed_bytes, dtype=np.uint8)
    unpacked = np.unpackbits(packed_np, bitorder='big')[:num_elements]
    if shape is not None:
        unpacked = unpacked.reshape(shape)
    return unpacked.astype(np.uint8)

def reconstruct_tensor(planes_dict: Dict[int, bytes], selected_bits: int, original_shape: Tuple[int, ...], device: Union[str, torch.device] = 'cpu') -> torch.Tensor:
    assert 1 <= selected_bits <= 16, f"selected_bits must be between 1 and 16, got {selected_bits}"
    num_elements = int(np.prod(original_shape)) if len(original_shape) > 0 else 1
    accum = np.zeros(num_elements, dtype=np.uint32)
    start_bit = 15
    end_bit = 16 - selected_bits
    for b in range(start_bit, end_bit - 1, -1):
        if b in planes_dict:
            bits = unpack_bit_plane(planes_dict[b], num_elements)
            accum |= (bits.astype(np.uint32) << b)
    uint16_arr = accum.astype(np.uint16).reshape(original_shape)
    return uint16_to_float16_torch(uint16_arr, device=device)
''')

with open("upr/converter.py", "w", encoding="utf-8") as f:
    f.write('''import os
import json
import torch
import numpy as np
from typing import Union, Optional
from tqdm import tqdm
from transformers import AutoModelForCausalLM
from .bit_ops import float16_to_uint16_numpy, extract_bit_plane_np, pack_bit_plane

def convert_to_bitplanes(model_or_path: Union[str, torch.nn.Module], output_directory: str, torch_dtype: torch.dtype = torch.float16) -> str:
    os.makedirs(output_directory, exist_ok=True)
    tensors_dir = os.path.join(output_directory, "tensors")
    os.makedirs(tensors_dir, exist_ok=True)
    if isinstance(model_or_path, str):
        model_name = model_or_path
        print(f"Loading Hugging Face model from: {model_name}")
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch_dtype, low_cpu_mem_usage=True)
    else:
        model_name = getattr(model_or_path, "name_or_path", "custom_model")
        model = model_or_path
    state_dict = model.state_dict()
    metadata = {"model_name_or_path": model_name, "num_tensors": len(state_dict), "tensors": {}}
    print(f"Converting {len(state_dict)} tensors to bit-plane format in '{output_directory}'...")
    for idx, (tensor_name, tensor) in enumerate(tqdm(state_dict.items(), desc="BitPlane Conversion")):
        tensor_folder_name = f"tensor_{idx}"
        tensor_folder_path = os.path.join(tensors_dir, tensor_folder_name)
        os.makedirs(tensor_folder_path, exist_ok=True)
        original_shape = list(tensor.shape)
        dtype_str = str(tensor.dtype).replace("torch.", "")
        uint16_arr = float16_to_uint16_numpy(tensor)
        planes_meta = {}
        for bit_idx in range(16):
            plane_filename = f"plane{bit_idx}.bin"
            plane_path = os.path.join(tensor_folder_path, plane_filename)
            bit_arr = extract_bit_plane_np(uint16_arr, bit_idx)
            packed_bytes = pack_bit_plane(bit_arr)
            with open(plane_path, "wb") as f:
                f.write(packed_bytes)
            planes_meta[str(bit_idx)] = f"tensors/{tensor_folder_name}/{plane_filename}"
        metadata["tensors"][tensor_name] = {"shape": original_shape, "dtype": dtype_str, "numel": int(tensor.numel()), "folder": f"tensors/{tensor_folder_name}", "planes": planes_meta}
    metadata_path = os.path.join(output_directory, "metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"Successfully converted model to BitPlane format at: {output_directory}")
    return output_directory
''')

with open("upr/loader.py", "w", encoding="utf-8") as f:
    f.write('''import os
import json
import torch
from typing import Optional, Union, Dict, Any
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoConfig
from .bit_ops import reconstruct_tensor

class BitPlaneModel:
    @classmethod
    def load_reconstructed_state_dict(cls, bitplane_directory: str, bits: int = 16, device: Union[str, torch.device] = 'cpu') -> Dict[str, torch.Tensor]:
        assert 1 <= bits <= 16, f"bits must be between 1 and 16, got {bits}"
        metadata_path = os.path.join(bitplane_directory, "metadata.json")
        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"metadata.json not found in '{bitplane_directory}'")
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        reconstructed_state_dict = {}
        tensors_meta = metadata["tensors"]
        start_bit = 15
        end_bit = 16 - bits
        for tensor_name, info in tqdm(tensors_meta.items(), desc=f"Reconstructing ({bits}-bit)"):
            original_shape = tuple(info["shape"])
            planes_dict = {}
            for b in range(start_bit, end_bit - 1, -1):
                plane_rel_path = info["planes"][str(b)]
                plane_full_path = os.path.join(bitplane_directory, plane_rel_path)
                if os.path.exists(plane_full_path):
                    with open(plane_full_path, "rb") as pf:
                        planes_dict[b] = pf.read()
            recon_tensor = reconstruct_tensor(planes_dict=planes_dict, selected_bits=bits, original_shape=original_shape, device=device)
            reconstructed_state_dict[tensor_name] = recon_tensor
        return reconstructed_state_dict

    @classmethod
    def from_pretrained(cls, bitplane_directory: str, bits: int = 16, base_model_id: Optional[str] = None, device_map: Optional[Union[str, Dict[str, Any]]] = None, torch_dtype: torch.dtype = torch.float16, **kwargs) -> torch.nn.Module:
        metadata_path = os.path.join(bitplane_directory, "metadata.json")
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        model_name = base_model_id or metadata.get("model_name_or_path")
        print(f"Instantiating model base architecture '{model_name}' for precision bits={bits}...")
        config = AutoConfig.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_config(config, torch_dtype=torch_dtype)
        state_dict = cls.load_reconstructed_state_dict(bitplane_directory=bitplane_directory, bits=bits, device='cpu')
        model.load_state_dict(state_dict, strict=True)
        if device_map is not None:
            model = model.to(device_map)
        return model
''')

with open("upr/metrics.py", "w", encoding="utf-8") as f:
    f.write('''import torch
import numpy as np
from typing import Dict, Any

def compute_weight_metrics(original: torch.Tensor, reconstructed: torch.Tensor) -> Dict[str, Any]:
    orig_f32 = original.detach().cpu().to(torch.float32)
    recon_f32 = reconstructed.detach().cpu().to(torch.float32)
    is_exact = bool(torch.equal(original.detach().cpu(), reconstructed.detach().cpu()))
    diff = torch.abs(orig_f32 - recon_f32)
    mae = float(diff.mean().item())
    rmse = float(torch.sqrt(torch.mean((orig_f32 - recon_f32) ** 2)).item())
    max_error = float(diff.max().item())
    orig_flat = orig_f32.view(-1)
    recon_flat = recon_f32.view(-1)
    norm_orig = torch.norm(orig_flat)
    norm_recon = torch.norm(recon_flat)
    if norm_orig == 0 or norm_recon == 0:
        cos_sim = 1.0 if norm_orig == norm_recon else 0.0
    else:
        cos_sim = float((torch.dot(orig_flat, recon_flat) / (norm_orig * norm_recon)).item())
    return {"exact_match": is_exact, "mae": mae, "rmse": rmse, "max_error": max_error, "cosine_similarity": cos_sim, "num_elements": int(orig_f32.numel())}
''')

# Install dependencies
!pip install -q transformers accelerate huggingface_hub torch numpy tqdm

from huggingface_hub import login
login(token=HF_TOKEN)

import torch
import numpy as np
import upr
print(f'UPR package successfully bootstrapped & imported! Version: {upr.__version__}')
print(f'PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

Mounted at /content/drive
Google Drive mounted successfully.
Active Working Directory: /content/drive/MyDrive/UniversalPrecisionRuntime


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


UPR package successfully bootstrapped & imported! Version: 0.1.0
PyTorch version: 2.11.0+cu128, CUDA available: True


### Step 2: Download Original Hugging Face FP16 Baseline Model
We load `Qwen/Qwen3.5-0.8B` in FP16 precision.

In [2]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3.5-0.8B'
print(f'Loading baseline model {MODEL_ID} in FP16...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
original_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

param_count = sum(p.numel() for p in original_model.parameters())
fp16_bytes = param_count * 2

baseline_stats = {
    'model_id': MODEL_ID,
    'num_parameters': param_count,
    'fp16_size_mb': fp16_bytes / (1024 * 1024),
    'num_tensors': len(original_model.state_dict())
}

print(f'Model loaded: {param_count:,} parameters ({fp16_bytes / (1024**2):.2f} MB in FP16)')
with open('results/original.json', 'w') as f:
    json.dump(baseline_stats, f, indent=2)
print('Saved baseline stats to results/original.json')

Loading baseline model Qwen/Qwen3.5-0.8B in FP16...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Model loaded: 752,393,024 parameters (1435.08 MB in FP16)
Saved baseline stats to results/original.json


### Step 3: Convert FP16 Model to BitPlane Checkpoint
Run `upr.convert_to_bitplanes()` to extract 16 binary bit planes per parameter tensor.

In [3]:
BITPLANE_DIR = 'models/bitplane_qwen'

print(f'Converting {MODEL_ID} -> {BITPLANE_DIR}...')
output_path = upr.convert_to_bitplanes(
    model_or_path=original_model,
    output_directory=BITPLANE_DIR,
    torch_dtype=torch.float16
)

# Check storage metadata
with open(os.path.join(BITPLANE_DIR, 'metadata.json'), 'r') as f:
    meta = json.load(f)

print(f'Successfully created BitPlane checkpoint with {meta["num_tensors"]} tensors.')

Converting Qwen/Qwen3.5-0.8B -> models/bitplane_qwen...
Converting 321 tensors to bit-plane format in 'models/bitplane_qwen'...


BitPlane Conversion: 100%|██████████| 321/321 [01:16<00:00,  4.21it/s]


Successfully converted model to BitPlane format at: models/bitplane_qwen
Successfully created BitPlane checkpoint with 321 tensors.


### Step 4: Level 1 Weight Reconstruction & Exact Equality Validation
Reconstruct all 16 planes (`bits=16`) and check `torch.equal()` against original weight tensors.

In [4]:
print('Reconstructing FP16 state dict from all 16 bit-planes (bits=16)...')
recon_state_dict = upr.BitPlaneModel.load_reconstructed_state_dict(
    bitplane_directory=BITPLANE_DIR,
    bits=16,
    device='cpu'
)

orig_state_dict = original_model.state_dict()

total_tensors = len(orig_state_dict)
exact_matches = 0
tensor_metrics = {}

for name, orig_tensor in orig_state_dict.items():
    recon_tensor = recon_state_dict[name]
    m = upr.compute_weight_metrics(orig_tensor, recon_tensor)
    tensor_metrics[name] = m
    if m['exact_match']:
        exact_matches += 1

match_percentage = (exact_matches / total_tensors) * 100
print('\n' + '='*60)
print(f'LEVEL 1 RECONSTRUCTION RESULTS (16-bit Full Reconstruction)')
print(f'Total Parameter Tensors Verified: {total_tensors}')
print(f'Exact Bitwise Matches (torch.equal == True): {exact_matches} / {total_tensors} ({match_percentage:.2f}%)')
print('='*60)

assert exact_matches == total_tensors, 'FAILED: BitPlane reconstruction is not 100% bit-exact!'

Reconstructing FP16 state dict from all 16 bit-planes (bits=16)...


Reconstructing (16-bit): 100%|██████████| 321/321 [01:13<00:00,  4.40it/s]



LEVEL 1 RECONSTRUCTION RESULTS (16-bit Full Reconstruction)
Total Parameter Tensors Verified: 321
Exact Bitwise Matches (torch.equal == True): 321 / 321 (100.00%)


### Step 5: Save Level 1 Evaluation Results
Save metric summaries to `results/full16.json`.

In [ ]:
level1_results = {
    'experiment': 'Full 16-Bit BitPlane Reconstruction',
    'total_tensors': total_tensors,
    'exact_matches': exact_matches,
    'match_percentage': match_percentage,
    'overall_status': 'PASSED' if exact_matches == total_tensors else 'FAILED',
    'tensor_details': tensor_metrics
}

with open('results/full16.json', 'w') as f:
    json.dump(level1_results, f, indent=2)

print('Successfully saved Level 1 results to results/full16.json!')

Successfully saved Level 1 results to results/full16.json!


: 